# D3 · 01 · Download ds004504 and build its source-space graphs

**Dataset 3** = OpenNeuro **ds004504** (Miltiadous et al., 2023; doi:10.18112/openneuro.ds004504, CC0): eyes-closed
resting-state EEG, 19 channels (10-20), 500 Hz. Used: the **36 AD and 29 HC** participants, from the provider's
**preprocessed derivatives**; the 23 FTD participants are excluded by design and never downloaded
(methodology log entry 44, `docs/dataset_d3.md`).

Processing (`configs/pipeline_d3.yaml`): the provider's 0.5-45 Hz filter, ASR and ICA cleaning are used as given
(nothing repeated). Recordings are split at the EEGLAB 'boundary' events that the provider's cleaning left
(data removed there): each continuous segment is resampled on its own and no window crosses a discontinuity. Added: the 10 s edge crop, resampling to 250 Hz, common average reference, the same 20 s /
50 % windows, window rejection, fsaverage source model, 68 Desikan-Killiany ROIs, bands, PLV / |ciPLV| and PSD
as D1/D2. Bad-channel rules are evaluated and reported only (no interpolation added).

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Code pin:** the setup cell runs the commit in `<outputs>/D3_CODE_COMMIT.txt` (Dataset 3's own pin); the
D1/D2 pin `CODE_COMMIT.txt` is never changed by D3 notebooks.

**Steps:** 1. download (≈ 2.2 GB to `My Drive/OpenNeuro/ds004504`, resumable, MD5-verified) ·
2. input check (36 AD / 29 HC, 19 channels in every file, EEGLAB boundary events per recording) · 3. smoke test (first subject) and its inspection ·
3b. real-data montage checks · 4. full graph construction (65 recordings, resumable; **only after you set
`CONFIRM_D3_FULL_BUILD = True`**) · 5. validation and completion report · 6. summary.
Output: `<outputs>/graphsD3_v4_<config hash>/` (never mixed with the D1/D2 graph runs). Experiments on D3:
`notebooks/d3/experiments/`. A GPU is not needed for this notebook.

In [ ]:
# ===== Dataset 3 setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_D3_CODE = False   # leave False. True only if you were asked to switch D3 to newer code
# Dataset 3 has its own code pin, <outputs>/D3_CODE_COMMIT.txt (all D3 notebooks run that commit);
# the D1/D2 pin CODE_COMMIT.txt is only reported, never read for checkout or changed here.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'D3']
     + (['--update-code'] if UPDATE_D3_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])

## 1. Download ds004504 (AD and HC derivatives only)

In [ ]:
sh("python scripts/d3_data.py download")

## 2. Input check (participants and recording headers; no signal is read)

In [ ]:
sh("python scripts/d3_data.py check")

## 3. Smoke test: first subject, then its inspection (diagnostics, not paper figures)

In [ ]:
sh("python scripts/build_graphs.py --config configs/pipeline_d3.yaml --limit 1")
sh("python scripts/inspect_graph_subjects.py --config configs/pipeline_d3.yaml --limit 1")
from IPython.display import Image, display
from adgraph.config import load_yaml
from adgraph.pipeline import graph_run_dir
smoke = graph_run_dir(OUT, load_yaml('configs/pipeline_d3.yaml'), 1)
for png in sorted(sorted(smoke.glob('inspection_*'))[-1].glob('*.png')):
    display(Image(str(png)))

## 3b. Real-data montage checks (must all PASS)
Explicitly selected inputs (never "the newest"): the D3 smoke run and the operator recorded in its manifest,
the frozen D1/D2 run by `graph_run_id 6d75de9975f0d9ae`, the header report of step 2. Checks: (1) electrode
labels and template positions of all 19 channels; (2) no confirmed flat channel entered reconstruction (a flat
channel stops its subject with `FlatChannelError`); (3) inverse operator 68 × 19 with the configured channels,
verified against the full MNE inverse; (4) 68 ROI names and order identical to D1/D2; (5) EEGLAB 'boundary'
discontinuities are handled (methodology log entry 45): the recording is split at every boundary, each continuous
segment is processed on its own and windows lie inside one segment, at least the margin from its ends. The check
runs the smoke subject and, in memory (nothing saved), the recording with the most boundary events, prints the
per-participant table (boundary events, segments, candidate windows, windows excluded for a discontinuity,
windows rejected by QC, accepted windows) and requires **zero accepted windows crossing a discontinuity or its
margin**. A failure stops the notebook here.

In [ ]:
sh("python scripts/d3_verify_smoke.py")

## 4. Full graph construction (resumable)
Starts only after you have reviewed steps 2-3b: set `CONFIRM_D3_FULL_BUILD = True` and run this cell (and the
ones below). Progress is saved per subject; if the runtime stops, *Run all* again.

In [ ]:
CONFIRM_D3_FULL_BUILD = False   # set True only after every check of step 3b passed and the smoke test was reviewed
if not CONFIRM_D3_FULL_BUILD:
    raise RuntimeError('Full D3 graph construction not started (CONFIRM_D3_FULL_BUILD = False). Review steps 2-3b, '
                       'then set CONFIRM_D3_FULL_BUILD = True and run this cell and the cells below.')
sh("python scripts/build_graphs.py --config configs/pipeline_d3.yaml --status")
sh("python scripts/build_graphs.py --config configs/pipeline_d3.yaml")

## 5. Validate the D3 graph dataset and write the completion report

In [ ]:
sh("python scripts/validate_graph_run.py --prefix graphsD3")

## 6. Summary

In [ ]:
import json
import pandas as pd
from adgraph.pipeline import latest_complete_graph_run
run = latest_complete_graph_run(OUT, 'graphsD3')
m = json.load(open(run / 'manifest.json'))
print('graph run:', run)
print('graph_run_id', m['graph_run_id'], '| commit', m['git_commit'], '| operator error',
      m['source_operator']['verification_rel_error'])
print('per dataset:', m['summary']['per_dataset'])
print('excluded by design:', len(m['registry'].get('excluded_by_design', {}).get('D3', [])), 'FTD participants')
s = pd.read_csv(run / 'subjects.csv')
display(s[[c for c in ['key', 'label', 'duration_s', 'median_std_uv', 'n_bad_channels', 'qc_spatial_rho', 'status']
          if c in s]])
print('discontinuities per participant (EEGLAB boundary events; windows of 20 s):')
display(s[[c for c in ['key', 'n_boundary_events', 'n_segments', 'n_segments_usable', 'usable_duration_s',
                       'n_windows_continuous_grid', 'n_windows_excluded_discontinuity', 'n_windows_total',
                       'n_windows_rejected', 'n_windows_kept', 'status'] if c in s]])
print(s.groupby('label')['n_windows_kept'].agg(['count', 'sum', 'median']))